# Extração de Dados Bitcoin - Workflow Macro

Script simplificado para uso em workflows do Databricks.
Extrai dados da API Coinbase, converte para BRL e salva em Delta Table.

In [0]:
import requests
from datetime import datetime

In [0]:
# Criar widget para receber parâmetros do pipeline (Key-Value)
# O valor será passado através da configuração Key-Value do pipeline
dbutils.widgets.text("api_key", "", "API Key CurrencyFreaks")

In [0]:
def extrair_dados_bitcoin():
    """Extrai o JSON completo da API da Coinbase."""
    url = 'https://api.coinbase.com/v2/prices/spot'
    resposta = requests.get(url)
    return resposta.json()

In [0]:
def extrair_cotacao_usd_brl():
    """Extrai a cotação USD-BRL da API CurrencyFreaks."""
    # Buscar API key dos parâmetros do pipeline (Key-Value)
    # Os valores configurados no pipeline são acessíveis via widgets
    api_key = dbutils.widgets.get("api_key")
    url = f'https://api.currencyfreaks.com/v2.0/rates/latest?apikey={api_key}'
    resposta = requests.get(url)
    return resposta.json()

In [0]:
def tratar_dados_bitcoin(dados_json, taxa_usd_brl):
    """Transforma os dados brutos da API, renomeia colunas, adiciona timestamp e converte para BRL."""
    valor_usd = float(dados_json['data']['amount'])
    criptomoeda = dados_json['data']['base']
    moeda_original = dados_json['data']['currency']
    
    # Convertendo de USD para BRL
    valor_brl = valor_usd * taxa_usd_brl
    
    # Adicionando timestamp como datetime object
    timestamp = datetime.now()
    
    dados_tratados = [{
        "valor_usd": valor_usd,
        "valor_brl": valor_brl,
        "criptomoeda": criptomoeda,
        "moeda_original": moeda_original,
        "taxa_conversao_usd_brl": taxa_usd_brl,
        "timestamp": timestamp
    }]
    
    return dados_tratados

In [0]:
# Extraindo dados
Dados_bitcoin = extrair_dados_bitcoin()
dados_cotacao = extrair_cotacao_usd_brl()

# Extraindo a taxa de conversão USD-BRL
taxa_usd_brl = float(dados_cotacao['rates']['BRL'])

# Tratando os dados e convertendo para BRL
dados_bitcoin_tratado = tratar_dados_bitcoin(Dados_bitcoin, taxa_usd_brl)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS pipe_api_bitcoin
COMMENT 'Catálogo de demonstração criado para o workshop de pipeline_api_bitcoin';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipe_api_bitcoin.bitcoin_data
COMMENT 'Schema para armazenar dados de Bitcoin processados';

In [0]:
# Criar Spark DataFrame
df = spark.createDataFrame(dados_bitcoin_tratado)

In [0]:
# Caminho da tabela Delta no Unity Catalog
delta_table_path = "pipe_api_bitcoin.bitcoin_data.bitcoin_data"

# Salvar como Delta Table (modo append se a tabela já existir)
df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(delta_table_path)